In [ ]:
# ================================
# Project Configuration
# ================================
from pathlib import Path
import os, random
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download

pd.set_option("display.max_columns", None)

SUBSAMPLE_SEED = 42
def set_seed(seed=SUBSAMPLE_SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

PROJECT_ROOT = Path("/kaggle/working")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
for folder in [RAW_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
# ================================
# Download IndicXNLI
# ================================
REPO_ID = "Divyanshu/indicxnli"
FILES = [
    "forward/train/xnli_hi.json", "forward/dev/xnli_hi.json", "forward/test/xnli_hi.json",
    "forward/train/xnli_te.json", "forward/dev/xnli_te.json", "forward/test/xnli_te.json",
]
for file in FILES:
    path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=file, local_dir=RAW_DIR)
    print(path)

In [ ]:
# ================================
# Loader
# ================================
import json
def load_xnli(language, split):
    path = RAW_DIR / "forward" / split / f"xnli_{language}.json"
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    records = next(iter(data.values()))
    df = pd.DataFrame(records)
    df["language"] = language
    df["split"] = split
    return df

hi_train, hi_dev, hi_test = load_xnli("hi","train"), load_xnli("hi","dev"), load_xnli("hi","test")
te_train, te_dev, te_test = load_xnli("te","train"), load_xnli("te","dev"), load_xnli("te","test")

print(hi_train.shape, te_train.shape)

In [ ]:
train_df = pd.concat([hi_train, te_train], ignore_index=True)
valid_df = pd.concat([hi_dev, te_dev], ignore_index=True)
test_df  = pd.concat([hi_test, te_test], ignore_index=True)

for df in [hi_train, hi_dev, hi_test, te_train, te_dev, te_test]:
    df.reset_index(drop=True, inplace=True)
    df["sample_id"] = df.index
    df["dataset"] = "IndicXNLI"

hi_train["stratify_key"] = hi_train["label"]
te_train["stratify_key"] = te_train["label"]

In [ ]:
# ================================
# Nested stratified budgeted subsets — NOW INCLUDES 2000 and 20000
# ================================
from sklearn.model_selection import StratifiedShuffleSplit

for df in [hi_train, hi_dev, hi_test, te_train, te_dev, te_test]:
    df.reset_index(drop=True, inplace=True)
    df["sample_id"] = df.index
    df["dataset"] = "IndicXNLI"

def create_nested_subsets(df, budgets, seed=42):
    budgets = sorted(budgets, reverse=True)
    subsets = {}
    current = df.copy()
    for budget in budgets:
        if budget >= len(current):
            subsets[budget] = current.copy()
            continue
        splitter = StratifiedShuffleSplit(n_splits=1, train_size=budget, random_state=seed)
        idx, _ = next(splitter.split(current, current["label"]))
        current = current.iloc[idx].reset_index(drop=True)
        subsets[budget] = current.copy()
    return dict(sorted(subsets.items()))

# UPDATED: added 2000 and 20000
TRAIN_BUDGETS = [50, 100, 500, 1000, 2000, 20000]

hi_subsets = create_nested_subsets(hi_train, TRAIN_BUDGETS, seed=42)
te_subsets = create_nested_subsets(te_train, TRAIN_BUDGETS, seed=42)

print("Hindi subset sizes:", {k: len(v) for k, v in hi_subsets.items()})
print("Telugu subset sizes:", {k: len(v) for k, v in te_subsets.items()})

In [ ]:
# ================================
# Save to parquet
# ================================
HI_DIR = PROCESSED_DIR / "hi"
TE_DIR = PROCESSED_DIR / "te"
HI_DIR.mkdir(parents=True, exist_ok=True)
TE_DIR.mkdir(parents=True, exist_ok=True)

for budget, subset in hi_subsets.items():
    subset.to_parquet(HI_DIR / f"train_{budget}.parquet", index=False)
hi_dev.to_parquet(HI_DIR / "valid.parquet", index=False)
hi_test.to_parquet(HI_DIR / "test.parquet", index=False)

for budget, subset in te_subsets.items():
    subset.to_parquet(TE_DIR / f"train_{budget}.parquet", index=False)
te_dev.to_parquet(TE_DIR / "valid.parquet", index=False)
te_test.to_parquet(TE_DIR / "test.parquet", index=False)

print(os.listdir(HI_DIR))

In [ ]:
import json

manifest = {
    "dataset": "IndicXNLI", "languages": ["hi","te"], "budgets": TRAIN_BUDGETS,
    "seed": 42, "sampling": "Nested Stratified", "stratification": "label",
}
with open(PROCESSED_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

In [ ]:
# ================================
# Manifest
# ================================
manifest = {
    "dataset": "IndicXNLI", "languages": ["hi","te"], "budgets": TRAIN_BUDGETS,
    "seed": 42, "sampling": "Nested Stratified", "stratification": "label",
}
with open(PROCESSED_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("Done. Now: Save Version → Save & Run All (Commit).")